# Milestone 3 — GNN Evaluation & Baseline Comparison
**Author:** Aaron  
**Course:** CSCI 3834, Winter 2026  

This notebook covers Aaron's deliverable for Milestone 3:
- Evaluate GNN results produced by Abdi's training script
- Compare GNN against Traditional ML baselines (RF, ET, XGB, LR, SVM)
- Compute AUC-PR, AUC-ROC, F1, Precision, Recall, Confusion Matrix
- Produce plots and an aggregated comparison table


# 1. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'torch-geometric', 'xgboost', 'scikit-learn',
                'pandas', 'numpy', 'matplotlib', 'seaborn'], check=True)
print('All dependencies installed.')

# 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d, Dropout
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
print('Imports done. PyTorch version:', torch.__version__)

# 3. Load & Preprocess Dataset

In [ ]:
# Adjust path if running locally
FILE_PATH = '/content/Fraud Detection Transactions Dataset.csv'

dataset = pd.read_csv(FILE_PATH)
print(f'Dataset shape: {dataset.shape}')
dataset.head(2)

In [ ]:
# Preprocessing mirrors Traditional_ML.ipynb
X = dataset.loc[:, 'Transaction_ID':'Is_Weekend'].copy()
y = dataset['Fraud_Label'].copy()

X.drop(columns=['Transaction_ID', 'User_ID', 'Risk_Score'], inplace=True)

X['Timestamp'] = pd.to_datetime(X['Timestamp'])
X['Hour']      = X['Timestamp'].dt.hour
X['Day']       = X['Timestamp'].dt.day
X['Month']     = X['Timestamp'].dt.month
X['DayOfWeek'] = X['Timestamp'].dt.dayofweek
X.drop(columns=['Timestamp'], inplace=True)

categorical_cols = ['Transaction_Type', 'Device_Type', 'Location',
                    'Merchant_Category', 'Card_Type', 'Authentication_Method']
X = pd.get_dummies(X, columns=categorical_cols)

print(f'Feature matrix shape: {X.shape}')
print(f'Class distribution:\n{y.value_counts()}')

In [ ]:
# Train / Val / Test split: 80 / 10 / 10, stratified
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, shuffle=True, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, shuffle=True,
    stratify=y_temp, random_state=RANDOM_STATE)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# 4. Baseline Traditional ML Models

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'F1'       : round(f1_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_true, y_pred, zero_division=0), 4),
        'AUC-ROC'  : round(roc_auc_score(y_true, y_prob), 4),
        'AUC-PR'   : round(average_precision_score(y_true, y_prob), 4),
    }

In [ ]:
results = {}

print('Training Random Forest ...')
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
results['Random Forest'] = compute_metrics(y_test, rf.predict(X_test), rf.predict_proba(X_test)[:, 1])
print('  Done.')

print('Training Extra Trees ...')
et = ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
et.fit(X_train, y_train)
results['Extra Trees'] = compute_metrics(y_test, et.predict(X_test), et.predict_proba(X_test)[:, 1])
print('  Done.')

print('Training XGBoost ...')
xgb = XGBClassifier(n_estimators=100, random_state=RANDOM_STATE,
                    use_label_encoder=False, eval_metric='logloss', verbosity=0)
xgb.fit(X_train, y_train)
results['XGBoost'] = compute_metrics(y_test, xgb.predict(X_test), xgb.predict_proba(X_test)[:, 1])
print('  Done.')

print('Training Logistic Regression ...')
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_sc, y_train)
results['Logistic Regression'] = compute_metrics(y_test, lr.predict(X_test_sc), lr.predict_proba(X_test_sc)[:, 1])
print('  Done.')

print('Training SVM (may take a few minutes) ...')
svm = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_sc, y_train)
results['SVM'] = compute_metrics(y_test, svm.predict(X_test_sc), svm.predict_proba(X_test_sc)[:, 1])
print('  Done.')

# 5. GNN Model (Graph Attention Network)

## 5.1 Build Transaction Graph

Each transaction is a **node**. Two transactions are connected by an edge if they share the same `User_ID`, creating a user-centric transaction graph. Node features are the preprocessed numeric/one-hot columns.

In [ ]:
raw = pd.read_csv(FILE_PATH)

user_groups = raw.groupby('User_ID').apply(lambda g: g.index.tolist()).to_dict()

edges_src, edges_dst = [], []
for indices in user_groups.values():
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            edges_src.extend([indices[i], indices[j]])
            edges_dst.extend([indices[j], indices[i]])  # undirected

edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
print(f'Nodes: {len(X)}  |  Edges: {edge_index.shape[1]:,}')

In [ ]:
X_scaled_all  = scaler.fit_transform(X)
node_features = torch.tensor(X_scaled_all, dtype=torch.float)
labels        = torch.tensor(y.values, dtype=torch.long)

data = Data(x=node_features, edge_index=edge_index, y=labels)

# Masks matching same stratified split
all_idx  = np.arange(len(y))
tr_idx, tmp = train_test_split(all_idx, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
vl_idx, ts_idx = train_test_split(tmp, test_size=0.50, stratify=y.iloc[tmp], random_state=RANDOM_STATE)

def make_mask(idx, n):
    m = torch.zeros(n, dtype=torch.bool); m[idx] = True; return m

data.train_mask = make_mask(tr_idx, len(y))
data.val_mask   = make_mask(vl_idx, len(y))
data.test_mask  = make_mask(ts_idx, len(y))
print(data)

## 5.2 Define GAT Architecture

In [ ]:
class FraudGAT(torch.nn.Module):
    '''
    Two-layer Graph Attention Network for binary fraud classification.
    Architecture: GAT(8 heads) -> BN -> Dropout -> GAT(1 head) -> BN -> Linear
    '''
    def __init__(self, in_channels, hidden=64, heads=8, dropout=0.3):
        super().__init__()
        self.conv1   = GATConv(in_channels, hidden, heads=heads, dropout=dropout)
        self.bn1     = BatchNorm1d(hidden * heads)
        self.conv2   = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout)
        self.bn2     = BatchNorm1d(hidden)
        self.dropout = Dropout(p=dropout)
        self.lin     = Linear(hidden, 2)

    def forward(self, x, edge_index):
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = self.dropout(x)
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        x = self.dropout(x)
        return self.lin(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 5.3 Train with Early Stopping

In [ ]:
# Class-imbalance weighting
n_neg = int((y.values == 0).sum())
n_pos = int((y.values == 1).sum())
class_weights = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float).to(device)

model     = FraudGAT(in_channels=data.num_node_features).to(device)
data      = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

def train_step():
    model.train(); optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); optimizer.step()
    return loss.item()

@torch.no_grad()
def eval_mask(mask):
    model.eval()
    out  = model(data.x, data.edge_index)
    prob = F.softmax(out, dim=1)[:, 1].cpu().numpy()
    pred = out.argmax(dim=1).cpu().numpy()
    yt   = data.y.cpu().numpy()
    m    = mask.cpu().numpy()
    return yt[m], pred[m], prob[m]

EPOCHS, PATIENCE = 200, 20
best_val_auc, patience_cnt, best_state = 0.0, 0, None
train_losses, val_aucs = [], []

for epoch in range(1, EPOCHS + 1):
    loss = train_step()
    train_losses.append(loss)
    yt, _, yp = eval_mask(data.val_mask)
    vauc = roc_auc_score(yt, yp)
    val_aucs.append(vauc)
    if vauc > best_val_auc:
        best_val_auc = vauc
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}. Best Val AUC-ROC: {best_val_auc:.4f}')
            break
    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d} | Loss: {loss:.4f} | Val AUC-ROC: {vauc:.4f}')

model.load_state_dict(best_state)
print('Training complete.')

# 6. GNN Evaluation on Test Set

In [ ]:
y_true_gnn, y_pred_gnn, y_prob_gnn = eval_mask(data.test_mask)

results['GNN (GAT)'] = compute_metrics(y_true_gnn, y_pred_gnn, y_prob_gnn)

print('GNN (GAT) — Test Set Metrics:')
for k, v in results['GNN (GAT)'].items():
    print(f'  {k}: {v}')

print('\nFull Classification Report:')
print(classification_report(y_true_gnn, y_pred_gnn, target_names=['Not Fraud', 'Fraud']))

# 7. Confusion Matrix — GNN

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_true_gnn, y_pred_gnn)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'], ax=ax)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix — GNN (GAT)')
plt.tight_layout()
plt.savefig('confusion_matrix_gnn.png', dpi=150)
plt.show()

# 8. Training Loss & Validation AUC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, color='steelblue')
axes[0].set_title('Training Loss per Epoch')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].grid(alpha=0.3)
axes[1].plot(val_aucs, color='darkorange')
axes[1].set_title('Validation AUC-ROC per Epoch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC-ROC'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('gnn_training_curves.png', dpi=150)
plt.show()

# 9. ROC & Precision-Recall Curves (All Models)

In [ ]:
probs_dict = {
    'Random Forest'      : rf.predict_proba(X_test)[:, 1],
    'Extra Trees'        : et.predict_proba(X_test)[:, 1],
    'XGBoost'            : xgb.predict_proba(X_test)[:, 1],
    'Logistic Regression': lr.predict_proba(X_test_sc)[:, 1],
    'SVM'                : svm.predict_proba(X_test_sc)[:, 1],
    'GNN (GAT)'          : y_prob_gnn,
}
y_test_np = y_test.values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#e377c2']

for (name, prob), color in zip(probs_dict.items(), colors):
    yt = y_true_gnn if name == 'GNN (GAT)' else y_test_np
    fpr, tpr, _ = roc_curve(yt, prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(yt,prob):.3f})', color=color)
    prec, rec, _ = precision_recall_curve(yt, prob)
    axes[1].plot(rec, prec, label=f'{name} (AP={average_precision_score(yt,prob):.3f})', color=color)

axes[0].plot([0,1],[0,1],'k--',lw=0.8)
axes[0].set_title('ROC Curves — All Models'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_title('Precision-Recall Curves — All Models'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150)
plt.show()

# 10. Aggregated Metrics Comparison Table

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
results_df = results_df[['Accuracy','Precision','Recall','F1','AUC-ROC','AUC-PR']]
results_df = results_df.sort_values('AUC-PR', ascending=False)

print('=== Model Comparison — Test Set ===')
display(results_df.style.highlight_max(axis=0, color='#d4efdf').format('{:.4f}'))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
results_df.plot(kind='bar', ax=ax, width=0.75, colormap='tab10', edgecolor='white')
ax.set_title('Model Performance Comparison — All Metrics')
ax.set_xlabel('Model'); ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=9); ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison_bar.png', dpi=150)
plt.show()

# 11. Summary

This notebook is Aaron's Milestone 3 deliverable for CSCI 3834.

| Task | Status |
|---|---|
| Preprocess data (consistent with Traditional_ML.ipynb) | ✅ |
| Retrain all baseline models for fair comparison | ✅ |
| Build transaction graph (user-shared edges) | ✅ |
| Implement 2-layer GAT with class-imbalance weighting | ✅ |
| Train with early stopping on AUC-ROC | ✅ |
| Compute AUC-PR, AUC-ROC, F1, Precision, Recall, Confusion Matrix | ✅ |
| ROC & Precision-Recall curves for all models | ✅ |
| Aggregated comparison table (sorted by AUC-PR) | ✅ |
| Bar chart visualization | ✅ |

**Key observations (fill in after running):**
- GNN (GAT) leverages user-relationship graph structure, which tree-based models cannot.
- AUC-PR is the primary metric given class imbalance (~68/32 split).
- GNN is expected to improve recall for the fraud class vs. baselines.
